In [1]:
import pdfplumber
import pandas as pd
import sqlite3
import os

In [5]:
import re

In [30]:
pdf = pdfplumber.open("class_008_Magical_Woolly_Mammoth_Sculpting.pdf")
full_text = "\n".join([page.extract_text() for page in pdf.pages])
pdf.close()

In [87]:
print(full_text)

Magical Woolly Mammoth Sculpting
Instructor: Location:
Professor Woolly Mammoth Northumberland
Course Type: Cost:
Fiber Arts £75.00
Learning Objectives
• Master the basics of needle felting to create wool sculptures
• Learn about different types of wool and their properties
• Discover techniques for shaping wool into realistic forms
• Develop artistic skills to add expressive details to sculptures
• Gain knowledge of historical significance of wool in art
Provided Materials
• Selection of colourful wool
• Felting needles
• Foam pad for felting
• Instruction booklet
• Mammoth-shaped templates
Skills Developed
Needle Felting Sculpting Artistic Expression Creative Crafting
Textile Knowledge
Course Description
Step into the whimsical world of 'Magical Woolly Mammoth Sculpting', where
creativity meets delightful eccentricity! In this unique fiber arts class, guided by the
illustrious Professor Woolly Mammoth, students at the School of Dandori will delve
into the playful art of needle feltin

In [ ]:
def extract_boxed_skills(page):
    """
    Extracts words from a specific page area and groups them into skills
    based on their physical distance from one another.
    """
    words = page.extract_words()

    # 1. Find the boundaries for the skills section
    try:
        header = next(w for w in words if "Skills" in w["text"])
        footer = next(
            w
            for w in words
            if "Course" in w["text"] and "Description" in page.extract_text()
        )

        # Filter words between "Skills Developed" and "Course Description"
        skill_words = [w for w in words if header["bottom"] < w["top"] < footer["top"]]
    except:
        return []

    if not skill_words:
        return []

    # 2. Group words into distinct skill phrases
    skills = []
    if skill_words:
        current_skill = [skill_words[0]["text"]]
        for i in range(1, len(skill_words)):
            prev = skill_words[i - 1]
            curr = skill_words[i]

            # Distance Logic:
            # If the vertical distance is small AND the horizontal gap is small,
            # they belong to the same skill box.
            v_gap = abs(curr["top"] - prev["top"])
            h_gap = curr["x0"] - prev["x1"]

            # Threshold: < 5px vertical shift and < 10px horizontal gap
            if v_gap < 5 and h_gap < 10:
                current_skill.append(curr["text"])
            else:
                skills.append(" ".join(current_skill))
                current_skill = [curr["text"]]

        skills.append(" ".join(current_skill))

    return [s for s in skills if s.strip()]


In [98]:
def extract_course_data(path):

    with pdfplumber.open(path) as pdf:

        full_text = "\n".join([page.extract_text() for page in pdf.pages])
        # Split the text into a list of cleaned lines
        lines = [line.strip() for line in full_text.split("\n") if line.strip()]

        def find_line_after(keyword, lines):
            """Finds the line immediately following the one containing the keyword."""
            for i, line in enumerate(lines):
                if keyword in line:
                    return lines[i + 1] if i + 1 < len(lines) else None
            return None

        # Handle the two-column interleaved layout
        # Line 1: 'Instructor: Location:'
        # Line 2: 'Chef Waffleby Harrogate'
        instructor_location_line = find_line_after("Instructor:", lines)
        course_type_cost_line = find_line_after("Course Type:", lines)

        # Extracting and splitting values
        # We use rsplit or specific logic if names/locations have spaces
        instructor = None
        location = None
        if instructor_location_line:
            # Assuming the instructor name is the first part and location is the last
            # Adjust 'Harrogate' as the anchor if it's always the same location
            parts = instructor_location_line.split(" ")
            location = parts[-1]  # 'Harrogate'
            instructor = " ".join(parts[:-1])  # 'Chef Waffleby'
        course_type = None
        cost = None
        if course_type_cost_line:
            # 'Culinary Arts £75.00' -> ['Culinary', 'Arts', '£75.00']
            parts = course_type_cost_line.split(" ")
            cost = parts[-1]  # '£75.00'
            course_type = " ".join(parts[:-1])  # 'Culinary Arts'

        data = {
            "class_id": (
                re.search(r"CLASS_\d+", full_text).group(0)
                if re.search(r"CLASS_\d+", full_text)
                else None
            ),  #
            "course_name": lines[0],  # 'The Art of Wondrous Waffle Weaving'
            "instructor": instructor,
            "course_type": course_type,
            "location": location,
            "cost": cost,
            # Objectives and Materials (Same logic as before)
            "learning_objectives": [
                re.sub(r"^•\s*", "", l)
                for l in lines
                if "•" in l and lines.index(l) < lines.index("Provided Materials")
            ], 
            "provided_materials": [
                re.sub(r"^•\s*", "", l)
                for l in lines
                if "•" in l and lines.index(l) > lines.index("Provided Materials")
            ],  
            "skills_developed": extract_boxed_skills(pdf.pages[1]) if len(pdf.pages) > 1 else [],  # Split by double space
            "course_description": " ".join(
                lines[
                    lines.index("Course Description")
                    + 1 : lines.index(next(l for l in lines if "Class ID" in l))
                ]
            ),  
        }

        return data

In [99]:
def process_pdf_folder(folder_path):
    """Iterates through a folder and returns a consolidated DataFrame."""
    all_data = []

    # List all PDF files in the directory
    files = [
        f
        for f in os.listdir(folder_path)
        if f.endswith(".pdf") and f.startswith("class_")
    ]

    for filename in files:
        path = os.path.join(folder_path, filename)
        print(f"Processing: {filename}...")
        try:
            row = extract_course_data(path)
            all_data.append(row)
        except Exception as e:
            print(f"Error in {filename}: {e}")
            break

    return pd.DataFrame(all_data)

In [100]:
df = process_pdf_folder("./course_pdfs")

Processing: class_001_The_Art_of_Wondrous_Waffle_Weaving.pdf...
Processing: class_008_Magical_Woolly_Mammoth_Sculpting.pdf...
Processing: class_015_the_magical_marmalade_adventure.pdf...
Processing: class_023_the_art_of_whimsical_waffle_weaving.pdf...


In [101]:
df.iloc[0]["skills_developed"]

['Culinary Arts',
 'Baking',
 'Creative Cooking',
 'Food Presentation',
 'Flavour Pairing']

In [90]:
def extract_boxed_skills(page):
    """
    Extracts words from a specific page area and groups them into skills
    based on their physical distance from one another.
    """
    words = page.extract_words()

    # 1. Find the boundaries for the skills section
    try:
        header = next(w for w in words if "Skills" in w["text"])
        footer = next(
            w
            for w in words
            if "Course" in w["text"] and "Description" in page.extract_text()
        )

        # Filter words between "Skills Developed" and "Course Description"
        skill_words = [w for w in words if header["bottom"] < w["top"] < footer["top"]]
    except (StopIteration, v):
        return []

    if not skill_words:
        return []

    # 2. Group words into distinct skill phrases
    skills = []
    if skill_words:
        current_skill = [skill_words[0]["text"]]
        for i in range(1, len(skill_words)):
            prev = skill_words[i - 1]
            curr = skill_words[i]

            # Distance Logic:
            # If the vertical distance is small AND the horizontal gap is small,
            # they belong to the same skill box.
            v_gap = abs(curr["top"] - prev["top"])
            h_gap = curr["x0"] - prev["x1"]

            # Threshold: < 5px vertical shift and < 10px horizontal gap
            if v_gap < 5 and h_gap < 10:
                current_skill.append(curr["text"])
            else:
                skills.append(" ".join(current_skill))
                current_skill = [curr["text"]]

        skills.append(" ".join(current_skill))

    return [s for s in skills if s.strip()]


def extract_course_data(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        # Full text for standard fields
        full_text = "\n".join([p.extract_text() for p in pdf.pages])
        lines = [l.strip() for l in full_text.split("\n") if l.strip()]

        # Skill-specific extraction from Page 2
        skills = extract_boxed_skills(pdf.pages[1]) if len(pdf.pages) > 1 else []

    # (Previous parsing logic for name, instructor, cost, etc. remains here)
    # ...

    return {
        "course_name": lines[1],
        "skills_developed": skills,  # Now returns ['Culinary Arts', 'Baking', 'Creative Cooking', etc.]
        # ... rest of your fields
    }

In [91]:
extract_course_data("course_pdfs/class_001_The_Art_of_Wondrous_Waffle_Weaving.pdf")

{'course_name': 'Instructor: Location:',
 'skills_developed': ['Culinary Arts',
  'Baking',
  'Creative Cooking',
  'Food Presentation',
  'Flavour Pairing']}

In [94]:
def extract_course_data(pdf_path):
    """Extracts structured data from a single Dandori Academy PDF."""
    with pdfplumber.open(pdf_path) as pdf:
        # Get all text for standard fields
        full_text = "\n".join(
            [page.extract_text() for page in pdf.pages if page.extract_text()]
        )
        lines = [line.strip() for line in full_text.split("\n") if line.strip()]

        # Get all words with coordinates for the skills section
        all_words = []
        for page in pdf.pages:
            all_words.extend(page.extract_words())

    # --- 1. Helper: Find value on the line below a label ---
    def get_val_below(label):
        for i, line in enumerate(lines):
            if label in line:
                return lines[i + 1] if i + 1 < len(lines) else ""
        return ""

    # --- 2. Handle Interleaved Header (Instructor/Location & Course Type/Cost) ---
    inst_loc_line = get_val_below("Instructor:")
    type_cost_line = get_val_below("Course Type:")

    instructor = " ".join(inst_loc_line.split()[:-1]) if inst_loc_line else None
    location = inst_loc_line.split()[-1] if inst_loc_line else None

    cost = type_cost_line.split()[-1] if type_cost_line else None
    course_type = " ".join(type_cost_line.split()[:-1]) if type_cost_line else None

    # --- 3. Extract Lists (Objectives & Materials) ---
    def extract_list(start_label, end_label):
        try:
            start_idx = next(i for i, v in enumerate(lines) if start_label in v)
            end_idx = next(i for i, v in enumerate(lines) if end_label in v)
            return [
                re.sub(r"^•\s*", "", lines[j]) for j in range(start_idx + 1, end_idx)
            ]
        except:
            return []

    # --- 4. Coordinate-Based Skills Extraction ---
    def get_boxed_skills(words):
        try:
            header_y = next(w for w in words if "Skills" in w["text"])["bottom"]
            footer_y = next(
                w for w in words if "Course" in w["text"] and w["top"] > header_y
            )["top"]
            skill_words = [w for w in words if header_y < w["top"] < footer_y]

            if not skill_words:
                return []

            grouped_skills = []
            current_phrase = [skill_words[0]["text"]]

            for i in range(1, len(skill_words)):
                prev, curr = skill_words[i - 1], skill_words[i]
                v_gap = abs(curr["top"] - prev["top"])
                h_gap = curr["x0"] - prev["x1"]

                # If they are on the same line (low v_gap) and close together (low h_gap)
                if v_gap < 4 and h_gap < 10:
                    current_phrase.append(curr["text"])
                else:
                    grouped_skills.append(" ".join(current_phrase))
                    current_phrase = [curr["text"]]

            grouped_skills.append(" ".join(current_phrase))
            return grouped_skills
        except:
            return []

    # --- 5. Course Description and ID ---
    try:
        desc_start = lines.index("Course Description") + 1
        desc_end = next(i for i, v in enumerate(lines) if "Class ID" in v)
        description = " ".join(lines[desc_start:desc_end])
    except:
        description = None

    class_id = (
        re.search(r"CLASS_\d+", full_text).group(0)
        if re.search(r"CLASS_\d+", full_text)
        else None
    )

    return {
        "class_id": class_id,
        "course_name": lines[1] if len(lines) > 1 else None,
        "instructor": instructor,
        "course_type": course_type,
        "location": location,
        "cost": cost,
        "learning_objectives": extract_list(
            "Learning Objectives", "Provided Materials"
        ),
        "provided_materials": extract_list("Provided Materials", "Skills Developed"),
        "skills_developed": get_boxed_skills(all_words),
        "course_description": description,
    }


def process_all_pdfs(folder_path):
    """Loops through folder and returns a consolidated DataFrame."""
    data_list = []
    for file in os.listdir(folder_path):
        if file.lower().endswith(".pdf"):
            print(f"Processing {file}...")
            try:
                data_list.append(extract_course_data(os.path.join(folder_path, file)))
            except Exception as e:
                print(f"Failed to process {file}: {e}")
    return pd.DataFrame(data_list)

In [95]:
df = process_all_pdfs("course_pdfs")

Processing class_001_The_Art_of_Wondrous_Waffle_Weaving.pdf...
Processing class_008_Magical_Woolly_Mammoth_Sculpting.pdf...
Processing class_015_the_magical_marmalade_adventure.pdf...
Processing class_023_the_art_of_whimsical_waffle_weaving.pdf...


In [96]:
df

,class_id,course_name,instructor,course_type,location,cost,learning_objectives,provided_materials,skills_developed,course_description
0,CLASS_4033,Instructor: Location:,Chef Waffleby,Culinary Arts,Harrogate,£75.00,[Master the technique of creating intricate wa...,"[Professional waffle iron, Selection of flours...","[The Art of Wondrous Waffle Weaving, Instructo...",Join us at the Harrogate Culinary Academy for ...
1,CLASS_8046,Instructor: Location:,Professor Woolly Mammoth,Fiber Arts,Northumberland,£75.00,[Master the basics of needle felting to create...,"[Selection of colourful wool, Felting needles,...","[Magical Woolly Mammoth Sculpting, Instructor:...",Step into the whimsical world of 'Magical Wool...
2,CLASS_0425,Instructor: Location:,Chef Marmalock,Culinary Arts,Inverness,£55.00,[Master the art of making traditional Scottish...,"[Citrus fruits (oranges, lemons), Unique ingre...","[The Magical Marmalade Adventure, Instructor:,...","Welcome to 'The Magical Marmalade Adventure,' ..."
3,CLASS_1041,Instructor: Location:,Chef Waffleton Crumble,Culinary Arts,York,£75.00,[Master the technique of creating intricate wa...,"[Waffle irons of various quirky shapes, Locall...","[The Art of Whimsical Waffle Weaving, Instruct...",Join us at the School of Dandori for 'The Art ...


In [109]:
df["cost"].apply(lambda c: pd.to_numeric(c[1:]))

0    75.0
1    75.0
2    55.0
3    75.0
Name: cost, dtype: float64